In [9]:
import sqlite3

conn = sqlite3.connect("urban_ai.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[('cbg_master',), ('pois',), ('cbg_poi_stats',), ('competitor_utility',), ('params',)]


In [10]:
# UTM Projection Function

def wgs84_to_utm_zone19n_nad83(latitude, longitude):
    a  = 6_378_137.0
    f  = 1 / 298.257222101
    e2 = 2 * f - f ** 2
    k0 = 0.9996
    E0 = 500_000.0

    lon0        = math.radians(-69.0)
    latitude_r  = math.radians(latitude)
    longitude_r = math.radians(longitude)

    N = a / math.sqrt(1 - e2 * math.sin(latitude_r) ** 2)
    T = math.tan(latitude_r) ** 2
    C = (e2 / (1 - e2)) * math.cos(latitude_r) ** 2
    A = math.cos(latitude_r) * (longitude_r - lon0)

    M = a * (
        (1 - e2/4 - 3*e2**2/64 - 5*e2**3/256) * latitude_r
        - (3*e2/8 + 3*e2**2/32 + 45*e2**3/1024) * math.sin(2 * latitude_r)
        + (15*e2**2/256 + 45*e2**3/1024) * math.sin(4 * latitude_r)
        - (35*e2**3/3072) * math.sin(6 * latitude_r)
    )

    x = E0 + k0 * N * (
        A
        + (1 - T + C) * A**3 / 6
        + (5 - 18*T + T**2 + 72*C - 58*(e2/(1-e2))) * A**5 / 120
    )

    y = k0 * (
        M + N * math.tan(latitude_r) * (
            A**2 / 2
            + (5 - T + 9*C + 4*C**2) * A**4 / 24
            + (61 - 58*T + T**2 + 600*C - 330*(e2/(1-e2))) * A**6 / 720
        )
    )

    return x, y

print("UTM projection function ready.")

UTM projection function ready.


In [11]:
# User Input Layer

print("=" * 60)
print("   HUFF ENGINE v2 - DATABASE POWERED")
print("=" * 60)

# Show available categories from database
print("\nAvailable categories:")
cursor.execute("""
    SELECT top_category, naics_code
    FROM pois
    GROUP BY top_category
    ORDER BY top_category
""")

rows = cursor.fetchall()
for i, row in enumerate(rows):
    print(f"  {str(i).rjust(2)}. {row[0]}  (NAICS: {row[1]})")

print("\nEnter store details:")
latitude_new  = float(input("  Latitude  (e.g., 42.27): "))
longitude_new = float(input("  Longitude (e.g., -71.80): "))
user_input    = input("  Category (name or NAICS code): ").strip()
store_size    = float(input("  Store size in sq meters: "))

# Look up alpha and beta from params table
cursor.execute("""
    SELECT alpha, beta, top_category
    FROM params
    WHERE top_category = ?
""", (user_input,))

row = cursor.fetchone()

if row is None:
    # Category not in params, use average values
    cursor.execute("SELECT AVG(alpha), AVG(beta) FROM params")
    avg = cursor.fetchone()
    alpha            = avg[0]
    beta             = avg[1]
    matched_category = user_input
    print(f"  Category not in params, using default alpha={alpha:.2f}, beta={beta:.2f}")
else:
    alpha            = row[0]
    beta             = row[1]
    matched_category = row[2]
    print(f"  Matched: '{matched_category}' | alpha={alpha} | beta={beta}")

print("Input received.")

   HUFF ENGINE v2 - DATABASE POWERED

Available categories:
   0. Accounting, Tax Preparation, Bookkeeping, and Payroll Services  (NAICS: 5412)
   1. Activities Related to Credit Intermediation  (NAICS: 522310)
   2. Activities Related to Real Estate  (NAICS: 531311)
   3. Administration of Economic Programs  (NAICS: 926120)
   4. Administration of Human Resource Programs  (NAICS: 9231)
   5. Advertising, Public Relations, and Related Services  (NAICS: 5418)
   6. Agencies, Brokerages, and Other Insurance Related Activities  (NAICS: 524210)
   7. Amusement Parks and Arcades  (NAICS: 713110)
   8. Automobile Dealers  (NAICS: 441120)
   9. Automotive Equipment Rental and Leasing  (NAICS: 532111)
  10. Automotive Parts, Accessories, and Tire Stores  (NAICS: 441320)
  11. Automotive Repair and Maintenance  (NAICS: 811122)
  12. Bakeries and Tortilla Manufacturing  (NAICS: 311811)
  13. Beer, Wine, and Liquor Stores  (NAICS: 445310)
  14. Beverage Manufacturing  (NAICS: 312130)
  15. Book S

In [12]:
# Cell 4: Calculate distance from new store to all CBGs
# NumPy vectorization

import numpy as np
import pandas as pd
import time
import math

start_time = time.time()

# Calculate new store UTM coordinates
utm_x_new, utm_y_new = wgs84_to_utm_zone19n_nad83(latitude_new, longitude_new)

# Fetch all CBG coordinates from database
cursor.execute("""
    SELECT cbg_id, utm_x, utm_y
    FROM cbg_master
""")

cbg_rows = cursor.fetchall()
cbgs_df  = pd.DataFrame(cbg_rows, columns=["cbg_id", "utm_x", "utm_y"])

# NumPy vectorized distance calculation (no loop, no apply!)
dx = cbgs_df["utm_x"].values - utm_x_new
dy = cbgs_df["utm_y"].values - utm_y_new
cbgs_df["dist_to_new"] = np.maximum(np.sqrt(dx**2 + dy**2), 1.0)

# Fetch pre-computed competitor utility for this category
cursor.execute("""
    SELECT cbg_id, sum_U_existing
    FROM competitor_utility
    WHERE top_category = ?
""", (matched_category,))

utility_rows = cursor.fetchall()
utility_df   = pd.DataFrame(utility_rows, columns=["cbg_id", "sum_U_existing"])

# Merge distances with pre-computed utilities
cbgs_df = cbgs_df.merge(utility_df, on="cbg_id", how="left")
cbgs_df["sum_U_existing"] = cbgs_df["sum_U_existing"].fillna(0)

print(f"CBGs loaded:   {len(cbgs_df)}")
print(f"Nearest CBG:   {cbgs_df['dist_to_new'].min():,.0f} m")
print(f"Farthest CBG:  {cbgs_df['dist_to_new'].max():,.0f} m")

CBGs loaded:   149
Nearest CBG:   382 m
Farthest CBG:  7,228 m


In [13]:
# Cell 5: Huff Model + Demand Estimation

# Calculate new store utility per CBG
cbgs_df["U_new"] = (store_size ** alpha) / (cbgs_df["dist_to_new"] ** beta)

# Calculate probability
total_U = cbgs_df["U_new"] + cbgs_df["sum_U_existing"]
cbgs_df["P_new"] = np.where(total_U > 0, cbgs_df["U_new"] / total_U, 0)

# Fetch historical demand for this category from database
cursor.execute("""
    SELECT s.cbg_id, SUM(s.visit_count) as total_demand
    FROM cbg_poi_stats s
    JOIN pois p ON s.placekey = p.placekey
    WHERE p.top_category = ?
    GROUP BY s.cbg_id
""", (matched_category,))

demand_rows = cursor.fetchall()
demand_df = pd.DataFrame(demand_rows, columns=["cbg_id", "total_demand"])

# Merge demand into CBGs
cbgs_df = cbgs_df.merge(demand_df, on="cbg_id", how="left")
cbgs_df["total_demand"]     = cbgs_df["total_demand"].fillna(0)
cbgs_df["predicted_visits"] = cbgs_df["P_new"] * cbgs_df["total_demand"]

total_visits = cbgs_df["predicted_visits"].sum()
print(f"Total Predicted Visits: {total_visits:,.1f}")

Total Predicted Visits: 29.1


In [14]:
# Cell 6: Results + Performance Benchmarking

print("\n" + "=" * 60)
print("   PREDICTION RESULTS")
print("=" * 60)
print(f"\n  Location:    ({latitude_new}, {longitude_new})")
print(f"  Category:    {matched_category}")
print(f"  Store Size:  {store_size:,.0f} sq meters")
print(f"  Alpha|Beta:  {alpha} | {beta}")
print(f"\n  >>> TOTAL PREDICTED VISITS: {total_visits:,.1f} <<<")

# Top 10 contributing neighborhoods
print("\n  Top 10 Contributing Neighborhoods:")
print(f"  {'CBG':<15} {'P_new':>8}  {'Demand':>8}  {'Predicted Visits':>16}")
print("  " + "-" * 52)

top = (cbgs_df[cbgs_df["total_demand"] > 0]
       .sort_values("predicted_visits", ascending=False)
       .head(10))

for _, row in top.iterrows():
    print(f"  {int(row['cbg_id']):<15} {row['P_new']:>8.4f}  "
          f"{row['total_demand']:>8.0f}  {row['predicted_visits']:>16.1f}")

end_time = time.time()
print(f"\n  Execution time (DB method): {end_time - start_time:.2f} seconds")


   PREDICTION RESULTS

  Location:    (42.27, -71.8)
  Category:    Beer, Wine, and Liquor Stores
  Store Size:  2,500 sq meters
  Alpha|Beta:  1.5 | 1.6

  >>> TOTAL PREDICTED VISITS: 29.1 <<<

  Top 10 Contributing Neighborhoods:
  CBG                P_new    Demand  Predicted Visits
  ----------------------------------------------------
  250277316002      0.0559        21               1.2
  250277307005      0.0200        58               1.2
  250277312031      0.0152        59               0.9
  250277301005      0.0141        62               0.9
  250277309022      0.0172        51               0.9
  250277320011      0.0160        53               0.8
  250277322023      0.0174        47               0.8
  250277307001      0.0167        48               0.8
  250277319002      0.0391        20               0.8
  250277311012      0.0133        58               0.8

  Execution time (DB method): 0.13 seconds
